# A DLR DMFT calculation

The goal of this notebook is to make a more economical DMFT calculation using the Discrete Lehman Representation of Green's functions (DLR). If you are interested in more of the theory behind this method,a good first reference is [(here)](https://doi.org/10.1103/PhysRevB.86.085133), which this tutorial follows closely. 

You will construct first a simple self-consistent loop for the Sachdev-Ye-Kitaev model, and then an IPT calculation with a more compact and economical framework than in Tutorial 1. 

## Discrete Lehman Representation
Here we take advantage of the spectral representation of the Green's function
\begin{equation}
G(\tau)=-\int_{-\infty}^{\infty} K(\tau, \omega) \rho(\omega) d \omega
\end{equation}
with the fermionic kernel being given by 
\begin{equation}
K(\tau, \omega)=\frac{e^{-\omega \tau}}{1+e^{-\beta \omega}}
\end{equation}
The kernel $K$ can be approximated by a low rank decomposition. This has some negative consequences, such as the ill-conditioning of analytic continuation (which you make have seen in the Pade approximates used in previous tutorials), but it also has some benefits. This behaviour can be exploited to provide a compact representation of $G$. 
\begin{equation}
G(\tau) \approx G_{\mathrm{DLR}}(\tau)=\sum_{k=1}^r \frac{e^{-\omega_k \tau}}{1+e^{-\omega_k}} \widehat{g}_k=\sum_{k=1}^r \widetilde{g}_k e^{-\omega_k \tau}
\end{equation}

### Constructing DLR Green's Functions
The DLR Green's function is constructed on $r$ Matsubara frequency points which are uniquely determined (independent of $G$) by a error $\epsilon$ in the accuracy of the representation and a high-energy cutoff. $\Lambda = \beta \omega_{max}$. 
These values are given to TRIQS via the parameters  `dlr_error` and `w_max` . In general, these should be convergence criterion. 
Particular care is needed when approximating stochastic data, such as quantum Monte Carlo data, which might include statistical noise.
 For the determinstic Dyson equations solved here, these should be made quite large and small respectively. Here you will see that the representation needs only $\mathcal{O}(10^2)$ points to converge correctly, compared to $\mathcal{O}(10^8)$ with a uniform Matsubara grid. 

 The example below shows how to construct a fermionic Green's function in both Matsubara frequencies and imaginary time. 

In [3]:
from triqs.gf import *
from triqs.gf.tools import *
from triqs.operators import *
from triqs.gf.block_gf import *
from triqs.gf.descriptors import Function
import h5
import numpy as np
from triqs.plot.mpl_interface import *
import matplotlib.pyplot as plt
import matplotlib as mpl
import json, sys, os


In [4]:
params = {
    "beta": 50.0,
    "mu": 0.0,
    "alpha" : 0.5,
    "w_max": 10.0,
    "dlr_err": 1e-8, 
    "J" : 1.0,
    "max_iter": 1000,
    "threshold": 1e-8,
}

In [5]:
gf_struct =[('up',1), ('dn',1)]
iw_mesh = MeshDLRImFreq(beta=params["beta"], statistic='Fermion', w_max= params["w_max"], eps = params["dlr_err"], symmetrize = True)
# careful! does not work without symmetrize=True
G_iw   = BlockGf(mesh=iw_mesh, gf_struct=gf_struct)
G_tau = make_gf_dlr_imtime(G_iw)
Sigma_iw = G_iw.copy()
Sigma_tau = make_gf_dlr_imtime(Sigma_iw)

### A word of caution
The DLR representation now allows one to evaluate $G(\tau)$ for any $\tau$, even if it is not included in the original grid. While these are faithful representations, the current TRIQS implementation does not impose additional properties of Green's functions, such as antiperiodicity. See the below examples for some subtleties in evaluating $G(\tau)$. 

In [ ]:
G_iw << SemiCircular(2)
G_tau << make_gf_dlr_imtime(G_iw)
G_dlr = make_gf_dlr(G_iw["up"]) # the DLR coefficents 

print(G_dlr(0.0)) # evaluating at tau = 0.0 for float
print(G_dlr(0))   # evaluating at iw_0 for int
print(G_dlr(50.0)) # evaluating at tau = beta gives density
print(G_dlr(45.0))  # evaluating at tau = 5.0
print(G_dlr(-5.0))  # evaluating at tau = -5.0 does not obey G(-tau) = -G(beta - tau)
print(G_dlr(-150.0)) # evaluating outside the range of the DLR coefficients does not wrap back around
print(G_dlr(150.0)) # evaluating outside the range of the DLR coefficients does not wrap back around


[[-0.50000001+2.1345501e-16j]]
[[0.00044346+2.40809328e-11j]]
[[-0.50000001-5.37912629e-16j]]
[[-0.06406086-4.89547088e-17j]]
[[-0.06406086-4.89547088e-17j]]
[[nan+nanj]]
[[nan+nanj]]


### Exercise 1 
Now we are going to evluate the self-consistent Dyson loop for the $O+1$D (impurity) Sachdev-Ye-Kitaev (SYK) model. The Hamiltonian is given by 
$$H = \sum^N_{ijkl} J_{ijkl} c^\dagger_i c^\dagger_j c_k c_l$$
where the $J_{ijkl}$ couplings between $N$ fermions that are random Gaussian couplings with constant variance $J^2$.
 This model has many interesting applications for non-Fermi liquids, or strange metals, and quantum criticality. You can read more about the model and its' extensions [(here)](https://doi-org.myaccess.library.utoronto.ca/10.1103/RevModPhys.94.035004). Here we are just interested in the simple form of the self-energy in the large-$N$ limit of the model which is exactly solvable. Here we have (up to some prefactors)
\begin{equation}
\Sigma(\tau) = -J^2 G^2(\tau)G(-\tau)
\end{equation}
which has a superficial resemble to the IPT equations (except with the local $G$ rather than the Weiss $\mathcal{G}_0$). Now it is your turn to write a self-consistent loop to solve for $G$. 

### Optional Exercise 1b
To really see the power of DLR, try converging these equations down to lower $T$. To do this, take a higher $T$ solution as a guess and continue lowering $T$. Steps of $\beta = 10, 20$ should be slow enough.  As you approach the $\beta = \infty$ limit, your result should begin to match the conformal solution to these equations. Try comparing!
\begin{equation}
G_c(\tau)=-\frac{\pi^{1 / 4}}{\sqrt{2 \beta}}\left(\sin \left(\frac{\pi \tau}{\beta}\right)\right)^{-1 / 2}
\end{equation}

### Exercise 2 

Now that you see the power of DLR, try repeating Tutorial 1 to converge the IPT equations for a given $\beta$ and $U$. See how much quicker your calculations are compared to Tutorial 1. 
